<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - Event Sequences/Useful Outcomes Discovery
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:20px;font-family:Arial"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial">
User interactions with banking services across web, mobile, chatbot, phone, and branch visits leave behind event trails. Most activity is routine (bill pay, viewing statements), but some journeys lead to valuable outcomes like product applications. Identifying these high-propensity journeys in real-time enables targeted campaigns and personalized offers.</p>
<p style="font-size:16px;font-family:Arial"> In this notebook we will explore the banking data available to us </p>



<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from teradataml import *
import getpass
import time

import pandas as pd
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
from collections import defaultdict
from typing import List, Dict, Tuple, Any
import colorsys
import numpy as np

from EventSequenceHelper import generate_color_palette, hex_to_rgba, generate_sankey_data, create_sankey_diagram
from event_flow_graph import generate_event_flow
from transition_matrix import generate_transition_matrix

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud Lake...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=0._Bank_ClickStream_-_Discovery.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'> <b> 2. Data exploring  </b></p>
<p style = 'font-size:16px;font-family:Arial'>Let us take a look at the data we have. </p>

In [ ]:
df = DataFrame(in_schema('DEMO_Bank','Session_Events'))
df

In [ ]:
df.shape

<p style = 'font-size:16px;font-family:Arial'>We have 2100000 rows of data. Now let us check how many events and count in each event. </p>

In [ ]:
value_counts = df.groupby('Event').count()
print(value_counts.head(100))

In [ ]:
def explore_each_target(training_table, classification_target):
    print("="*60 + "\nVisualizing "+classification_target+"\n" + "="*60)

    try:
        db_drop_table("npath_aggregated_outcome_paths")
    except:
        True

    try:
        db_drop_table("npath_aggregated_non_outcome_paths")
    except:
        True
        
    execute_sql("""
        create volatile table npath_aggregated_outcome_paths as
        (
            select outcome, page_path,  count(*) as counts
            from npath(
                on (select * from {0})
                    partition by UserId, SessionId
                   order by Event_TS
                using
                    mode(nonoverlapping)
                    pattern('A*.B')
                    Symbols(
                      Event not like 'Apply%' as A,
                      Event like '{1}' as B -- 'Apply%'
                    )
                    Result(
                      accumulate(CDISTINCT Event of ANY(A,B)) as page_path,
                      first(Event of ANY(B)) as outcome
                    )
                    Filter(
                      FIRST(Event_TS + interval '1' day OF ANY (A)) >
                      FIRST(Event_TS of ANY(B))
                    )
            ) as dt2
            group by 1,2
        ) with data on commit preserve rows
    """.format(training_table,classification_target)
    )

    execute_sql("""
    create volatile table npath_aggregated_non_outcome_paths as
    (
        select page_path,  count(*) as counts
        from npath(
            on (select * from {0})
                partition by UserId, SessionId
                order by Event_TS
            using
                mode(nonoverlapping)
                pattern('A*.B$')
                Symbols(
                  Event not like 'Apply%' as A,
                  Event not like 'Apply%' as B
                )
                Result(
                  accumulate(CDISTINCT Event of ANY(A,B)) as page_path
                )
                Filter(
                  FIRST(Event_TS + interval '1' day OF ANY (A)) >
                  LAST(Event_TS of ANY(B))
                )
        ) as dt2
        group by 1 
    ) with data on commit preserve rows
""".format(training_table)
    )

    df = DataFrame("npath_aggregated_outcome_paths").to_pandas().reset_index()
    df['page_path'] = df['page_path'].str.strip('[]').str.split(', ')
    outcome_page_path_data = df['page_path'].tolist()

    df = DataFrame("npath_aggregated_non_outcome_paths").to_pandas().reset_index()
    df['page_path'] = df['page_path'].str.strip('[]').str.split(', ')
    non_outcome_page_path_data = df['page_path'].tolist()

    fig_n = create_sankey_diagram(
        sequences=outcome_page_path_data,
        num_steps=7,
        top_n=15,
        min_flow=2,
        title="Banking User Journey for \""+classification_target+"\" Outcome",
        width=1100,
        height=800
    )
    fig_n.show()

    fig_n = create_sankey_diagram(
        sequences=non_outcome_page_path_data,
        num_steps=7,
        top_n=15,
        min_flow=2,
        title="Banking User Journey with Non-Outcome Paths",
        width=1100,
        height=800
    )
    fig_n.show()



In [ ]:
training_table = 'DEMO_Bank.Session_Events'

In [ ]:
##
## CHOOSE TARGET FOR CLASSIFICATION MODEL
##

classification_targets = ['ApplyCreditCard','ApplyAutoLoan','ApplyCheckingAccount','ApplyMortgage',\
                          'ApplyPersonalLoan','ApplySavingsAccount', 'ApplyTeenChecking']

for classification_target in classification_targets:
    explore_each_target(training_table, classification_target)

#Please be patient it take 3-4minutes for visualizations to appear

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'> <b> 3. Generating Interactive Transition Matrix with Joint and Conditional Probabilities  </b></p>

In [ ]:
output = generate_transition_matrix(
    df.to_pandas().reset_index(),
    user_col="UserID",
    session_col="SessionID",
    time_col="Event_TS",
    event_col="Event",
    output_path="transition_matrix.html",
)

<p style = 'font-size:16px;font-family:Arial'> The html is saved as transaction_matrix.html, please open it separatley by double-clicking it from the file browser and click trust html for data to display.</p>

<p style = 'font-size:20px;font-family:Arial'> <b> Generating Interactive Graph Visualization of Event Transitions  </b></p>

In [ ]:
output = generate_event_flow(
    df.to_pandas().reset_index(),
    user_col="UserID",
    session_col="SessionID",
    time_col="Event_TS",
    event_col="Event",
    output_path="event_flow.html",
)

<p style = 'font-size:16px;font-family:Arial'> The html is saved as event_flow.html, please open it separatley by double-clicking it from the file browser and click trust html for data to display</p>

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>4. Cleanup </b></p>

In [ ]:
remove_context()

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>